In [1]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass
from sklearn.metrics import accuracy_score, roc_auc_score
import torch
from torch import nn
from torch.nn import CrossEntropyLoss

from tfmplayground.model import NanoTabPFNModel
from tfmplayground.priors import PriorDumpDataLoader
from tfmplayground.train import train
from tfmplayground.utils import get_default_device
from tfmplayground.interface import NanoTabPFNClassifier
from tfmplayground.callbacks import ConsoleLoggerCallback, WandbLoggerCallback

from tfmplayground.evaluation import get_openml_predictions, TOY_TASKS_CLASSIFICATION, TABARENA_TASKS

/Users/elias/anaconda3/envs/tfm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@dataclass
class ModelConfig:
    num_attention_heads: int = 6
    embedding_size: int = 192
    mlp_hidden_size: int = 768
    num_layers: int = 6
    num_outputs: int = 10

@dataclass
class PriorConfig:
    filename: str = '../data/tabicl_4k_50x3.h5'
    num_steps: int = 25
    batch_size: int = 50

@dataclass
class TrainConfig:
    epochs: int = 5
    accumulate: int = 1
    lr: float = 1e-4
    multigpu: bool = False
    runname: str = 'nanotabpfn'
    workdir: str = '../logs/'

model_cfg = ModelConfig()
prior_cfg = PriorConfig()
train_cfg = TrainConfig()
device = get_default_device()

In [3]:
model = NanoTabPFNModel(
    **model_cfg.__dict__
)
criterion = CrossEntropyLoss()

In [4]:
prior = PriorDumpDataLoader(**prior_cfg.__dict__, device=device)

In [5]:
class ToyEvaluationLoggerCallback(ConsoleLoggerCallback):
    def __init__(self, tasks):
        self.tasks = tasks

    def on_epoch_end(self, epoch: int, epoch_time: float, loss: float, model, **kwargs):
        classifier = NanoTabPFNClassifier(model, device)
        predictions = get_openml_predictions(model=classifier, tasks=self.tasks)
        scores = []
        for dataset_name, (y_true, y_pred, y_proba) in predictions.items():
            scores.append(accuracy_score(y_true, y_pred))
        avg_score = sum(scores) / len(scores)
        print(f'epoch {epoch:5d} | time {epoch_time:5.2f}s | mean loss {loss:5.2f} | avg accuracy {avg_score:.3f}',
              flush=True)
class ProductionEvaluationLoggerCallback(WandbLoggerCallback):
    def __init__(self, project: str, name: str = None, config: dict = None, log_dir: str = None):
        super().__init__(project, name, config, log_dir)

    def on_epoch_end(self, epoch: int, epoch_time: float, loss: float, model, **kwargs):
        classifier = NanoTabPFNClassifier(model, device)
        print('openml evaluation...', flush=True)
        predictions = get_openml_predictions(
            max_n_samples = 1_000,
            max_n_features = 100,
            model=classifier, classification=True, tasks=TABARENA_TASKS
        )
        print('evaluation done.', flush=True)
        scores = []
        for dataset_name, (y_true, y_pred, y_proba) in predictions.items():
            scores.append(roc_auc_score(y_true, y_proba, multi_class='ovr'))
        avg_score = sum(scores) / len(scores)
        self.wandb.log({
            'epoch': epoch,
            'epoch_time': epoch_time,
            'mean_loss': loss,
            'tabarena_avg_roc_auc': avg_score
        })
        print(f'epoch {epoch:5d} | time {epoch_time:5.2f}s | mean loss {loss:5.2f} | avg roc auc {avg_score:.3f}',
              flush=True)

# callbacks = [ProductionEvaluationLoggerCallback('tfm', train_cfg.runname)]
callbacks = [ProductionEvaluationLoggerCallback('tfm')]
train_cfg.runname=callbacks[0].wandb.run.name
# callbacks = [ToyEvaluationLoggerCallback(TOY_TASKS_CLASSIFICATION)]

trained_model, loss = train(
    model=model,
    prior=prior,
    criterion=criterion,
    epochs=train_cfg.epochs,
    accumulate_gradients=train_cfg.accumulate,
    lr=train_cfg.lr,
    device=device,
    callbacks=callbacks,
    ckpt=None,
    multi_gpu=train_cfg.multigpu,
    run_name=train_cfg.runname,
    workdir=train_cfg.workdir,
)

wandb: Currently logged in as: eliasdubbeldam to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


openml evaluation...
evaluation done.
epoch     1 | time  9.31s | mean loss  1.14 | avg roc auc 0.599
openml evaluation...
evaluation done.
epoch     2 | time  8.82s | mean loss  0.77 | avg roc auc 0.571
openml evaluation...
evaluation done.
epoch     3 | time  8.64s | mean loss  0.67 | avg roc auc 0.442
openml evaluation...
evaluation done.
epoch     4 | time  8.75s | mean loss  0.67 | avg roc auc 0.491
openml evaluation...
evaluation done.
epoch     5 | time  8.70s | mean loss  0.66 | avg roc auc 0.453


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▃▅▆█
epoch_time,█▃▁▂▂
mean_loss,█▃▁▁▁
tabarena_avg_roc_auc,█▇▁▃▁
epoch,5
epoch_time,8.70201
mean_loss,0.66058
tabarena_avg_roc_auc,0.45279
